In [ ]:
from bs4 import BeautifulSoup
import re
import time
import json
from openai import OpenAI
client = OpenAI(api_key="EMPTY", base_url="http://localhost:9995/v1")

import sys
sys.path.insert(0, "..")
from pipeline.prompts import SYSTEM_PROMPT
from pipeline.data.xml_utils import get_entity_positions

In [ ]:
# generation task
def run_model(note, temperature=0, examples=None):
    global client
    
    examples_str = ""
    if examples is not None and examples.strip() != "":
        examples_list = json.loads(examples)
        examples_str_list = []
        for i, example in enumerate(examples_list):
            examples_str_list.append(
            f"""<example_note>
{example['original'].strip()}
</example_note>

<example_phi_entities>
{example['anonymized'].strip()}
</example_phi_entities>""")
        examples_str = "\n\n".join(examples_str_list)
        examples_str = f"<examples>\n{examples_str}\n</examples>\n\n"

    input_text = \
f"""<instructions>
{SYSTEM_PROMPT}
</instructions>

{examples_str}<note>
{note.strip()}
</note>

<phi_entities>
"""

    response = client.completions.create(
        model="../SFT/Qwen3-0.6B-Base_SFT",
        prompt=input_text,
        max_tokens=2048,
        temperature=temperature,
        seed=2025,
        stop=["</phi_entities>"]
    )
    
    response_text = "<phi_entities>\n" + response.choices[0].text
    response_text = response_text.split("</phi_entities>", 1)[0] + "</phi_entities>"
    response_text = response_text

    return response_text

In [ ]:
def simple_text_anonymization(native_text, xml_string):
    positions = get_entity_positions(native_text, xml_string)
    
    if not positions:
        return {
            'text': native_text,
            'entities': [{
                'entity': 'NAME',
                'score': 1.0,
                'index': 0,
                'word': '',
                'start': 0,
                'end': 0
            }]
        }, {}
    
    # Create anonymized text with replacements
    to_be_replaced = {}
    for pos in positions:
        content = pos['content']
        tag_type = pos['type']
        metadata = pos['metadata']
        
        # Create replacement value based on tag type and metadata
        if tag_type == 'name':
            id_val = metadata.get('id', '1')
            replacement = f"***NAME-{id_val}***"
        elif tag_type == 'date':
            # For dates, use formatted date if available
            formatted = metadata.get('formatted', content)
            replacement = f"***{formatted}***"
        else:
            id_val = metadata.get('id')
            if id_val:
                replacement = f"***{tag_type.upper()}-{id_val}***"
            else:
                replacement = f"***{tag_type.upper()}***"
        
        to_be_replaced[content] = replacement
    
    # Sort by length (longest first) to handle overlaps
    sorted_replacements = sorted(to_be_replaced.items(), key=lambda x: len(x[0]), reverse=True)
    
    # Apply replacements
    anonymized_text = native_text
    for original, replacement in sorted_replacements:
        pattern = re.escape(original)
        anonymized_text = re.sub(f'(?<!\\*){pattern}(?!\\*)', replacement, anonymized_text)
    
    # Create entity list for highlighting
    entities = []
    annotated_entities = {}
    
    for pos in positions:
        content = pos['content']
        tag_type = pos['type']
        metadata = pos['metadata']
        
        # Create replacement value
        if tag_type == 'name':
            id_val = metadata.get('id', '1')
            replacement = f"***NAME-{id_val}***"
        elif tag_type == 'date':
            formatted = metadata.get('formatted', content)
            replacement = f"***{formatted}***"
        else:
            id_val = metadata.get('id')
            if id_val:
                replacement = f"***{tag_type.upper()}-{id_val}***"
            else:
                replacement = f"***{tag_type.upper()}***"
        
        # Track entity mapping
        if replacement not in annotated_entities:
            annotated_entities[replacement] = [content]
        else:
            annotated_entities[replacement].append(content)
        
        # Find positions in anonymized text
        pattern = re.compile(re.escape(replacement))
        matches = pattern.finditer(anonymized_text)
        for match in matches:
            start = match.start()
            end = match.end()
            # Check if entity is not already annotated
            if not any(entity['start'] <= start and entity['end'] >= end for entity in entities):
                entities.append({
                    'entity': tag_type.upper(),
                    'score': 1.0,
                    'index': start,
                    'word': replacement,
                    'start': start,
                    'end': end
                })
    
    if len(entities) == 0:
        entities = [{
            'entity': 'NAME',
            'score': 1.0,
            'index': 0,
            'word': '',
            'start': 0,
            'end': 0
        }]
    
    return {
        'text': anonymized_text,
        'entities': entities
    }, annotated_entities


In [ ]:
def sanitize_note(note, examples=None):
    note = note.strip()
    if len(note) < 50:
        raise gr.Error("Note must be at least 50 characters long.")
    start_time = time.time()
    phi_entities = run_model(note, temperature=0, examples=examples)
    end_time = time.time()
    print(phi_entities)
    text, json_entities = simple_text_anonymization(note, phi_entities)
    return end_time - start_time, text, json_entities

In [ ]:
sample_examples = [
    {
        "original": """XR Chest 1 View Portable
CLINICAL INDICATION: Shortness of breath
COMPARISON: Radiograph dated 09/21 and 8/10/2023; CT of the chest dated January 10, 2025
FINDINGS:
Support Devices: None.
Heart/Mediastinum: Within normal limits.
Lungs/Pleura: Severe emphysematous changes of the upper lobes bilaterally. No focal consolidation, large pleural effusion, or pneumothorax. Minimal left basilar atelectasis.
Bones: No acute abnormality.
Other: None.
IMPRESSION: No acute cardiopulmonary abnormality.
The results were discussed with the primary provider,Bardia Khosravi on 09/19/2025 14:27.""",
        "title": "CXR Report (en)",
    },
    {
        "original": """Radiografía de Tórax 1 Vista Portátil
INDICACIÓN CLÍNICA: Disnea
COMPARACIÓN: Radiografía fechada 09/21 y 8/10/2023; TC de tórax fechada 10 de enero de 2025
HALLAZGOS:
Dispositivos de Soporte: Ninguno.
Corazón/Mediastino: Dentro de límites normales.
Pulmones/Pleura: Cambios enfisematosos severos de los lóbulos superiores bilateralmente. Sin consolidación focal, derrame pleural grande, o neumotórax. Atelectasia basilar izquierda mínima.
Huesos: Sin anormalidad aguda.
Otros: Ninguno.
IMPRESIÓN: Sin anormalidad cardiopulmonar aguda.
Los resultados fueron discutidos con el proveedor primario, Bardia Khosravi el 19/09/2025 14:27.""",
        "title": "CXR Report (es)",
    }
]

In [ ]:
# gradio app that takes a note and returns the anonymized note
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## EdgeDeID: Medical Note Anonymization on Edge Devices")
    
    # Add Examples Section
    with gr.Accordion("Few-shot Examples (Optional)", open=False):
        with gr.Row():
            with gr.Column():
                example1_input = gr.Textbox(label="Example 1 - Original Note", lines=3)
                example1_output = gr.Textbox(label="Example 1 - Anonymized Version", lines=3)
            with gr.Column():
                example2_input = gr.Textbox(label="Example 2 - Original Note", lines=3)
                example2_output = gr.Textbox(label="Example 2 - Anonymized Version", lines=3)
            with gr.Column():
                example3_input = gr.Textbox(label="Example 3 - Original Note", lines=3)
                example3_output = gr.Textbox(label="Example 3 - Anonymized Version", lines=3)
    
    # Main Input/Output Section
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Original Note")
            note_input = gr.Textbox(label="Note", lines=10)
            submit_button = gr.Button("Submit", variant="primary")
        with gr.Column():
            gr.Markdown("### Anonymized Note")
            time_taken = gr.Number(label="Time Taken (s)", interactive=False)
            note_output = gr.HighlightedText(label="Anonymized Note")
            entity_dict = gr.JSON(label="Entity Dictionary")
            
    with gr.Row():
        prefilled_samples = gr.Examples(
            examples=[sample['original'] for sample in sample_examples],
            inputs=[note_input],
            label = [sample['title'] for sample in sample_examples]
        )

    # Modified click handler to include examples
    def process_with_examples(note, ex1_in, ex1_out, ex2_in, ex2_out, ex3_in, ex3_out):
        examples = []
        # Add non-empty examples to the list
        for orig, anon in [(ex1_in, ex1_out), (ex2_in, ex2_out), (ex3_in, ex3_out)]:
            if orig and anon:  # Only add if both fields are non-empty
                orig = orig.strip()
                anon = anon.replace("<phi_entities>\n", "").replace("\n</phi_entities>", "").strip()
                examples.append({"original": orig, "anonymized": anon})
        
        examples_json = json.dumps(examples) if examples else ""
        return sanitize_note(note, examples=examples_json)

    submit_button.click(
        fn=process_with_examples,
        inputs=[
            note_input,
            example1_input,
            example1_output,
            example2_input,
            example2_output,
            example3_input,
            example3_output
        ],
        outputs=[time_taken, note_output, entity_dict]
    )
    
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    debug=True,
    inline=False,
)